# Parkville Skin & Hair Care Expert — Full Production-Grade RAG System

A **PDF-first, citation-grounded, hybrid Retrieval-Augmented Generation (RAG)** pipeline for the **Parkville Skin & Hair Care Expert** domain.

### What this notebook delivers

- Uploads **two PDF knowledge-base files** (Skin Care + Hair Care)
- Extracts text **page-by-page** with source traceability
- Detects **product boundaries and semantic sections**
- Builds **product-aware chunks** with rich metadata
- Creates a persistent **dense vector index (Chroma)**
- Creates a **sparse BM25 index** for exact product/ingredient matches
- Performs **Hybrid Retrieval → RRF Fusion → Cross-Encoder Reranking**
- Uses **query understanding + metadata-aware retrieval**
- Applies a **confidence gate** before generation
- Generates **strict JSON answers** grounded only in retrieved evidence
- Enforces **citation validity** and **pregnancy-safety label integrity**
- Supports **out-of-scope refusal**
- Ships with a **20-question Golden Evaluation Set**
- Measures retrieval + generation + safety + latency metrics
- Exports the index manifest and evaluation reports

> **Important:** The PDF files are the real runtime inputs. The Markdown versions supplied with the project were used only to understand the source structure while engineering this notebook; the pipeline itself uploads and processes PDFs directly.


## 0 — Architecture

```text
                         ┌─────────────────────┐
                         │   2 PDF Knowledge   │
                         │  Bases (Skin/Hair)  │
                         └──────────┬──────────┘
                                    │
                                    ▼
                         ┌─────────────────────┐
                         │ PDF Text Extraction │
                         │ page-level lineage  │
                         └──────────┬──────────┘
                                    │
                                    ▼
                         ┌─────────────────────┐
                         │ Product-aware       │
                         │ Parsing + Chunking  │
                         └──────────┬──────────┘
                                    │
                    ┌───────────────┴───────────────┐
                    ▼                               ▼
             Dense Embeddings                    BM25
             (SentenceTransformer)              Sparse Index
                    │                               │
                    └───────────────┬───────────────┘
                                    ▼
                            Hybrid Retrieval
                                    │
                                    ▼
                             RRF Fusion
                                    │
                                    ▼
                         Cross-Encoder Reranking
                                    │
                                    ▼
                           Confidence / Scope
                                Gate
                              /                               refuse       answer
                                    │
                                    ▼
                           Grounded LLM JSON
                                    │
                     ┌──────────────┼──────────────┐
                     ▼              ▼              ▼
                 Schema          Citation       Safety
                Validation       Guard          Guard
                     └──────────────┼──────────────┘
                                    ▼
                           Final Answer + Sources
                                    │
                                    ▼
                           Evaluation / Logging
```


## 1 — Environment & Configuration

In [4]:

# Colab / notebook environment
!pip install -q \
  pymupdf>=1.24 \
  sentence-transformers>=5.0 \
  chromadb>=1.0 \
  rank-bm25>=0.2.2 \
  openai>=1.90 \
  jsonschema>=4.24 \
  pandas>=2.2 \
  numpy>=1.26 \
  scikit-learn>=1.5 \
  tenacity>=9.0 \
  tqdm>=4.66 \
  rapidfuzz>=3.0


In [5]:

import os
import re
import json
import time
import math
import hashlib
import textwrap
import statistics
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

CONFIG = {
    # -----------------------------
    # Domain
    # -----------------------------
    "assistant_name": "Parkville Skin & Hair Care Expert",
    "allowed_categories": {"skin_care", "hair_care"},
    "max_question_chars": 600,

    # -----------------------------
    # PDF parsing / chunking
    # -----------------------------
    "min_page_chars": 80,
    "chunk_target_chars": 1200,
    "chunk_overlap_chars": 180,
    "max_chunk_chars": 1700,
    "min_chunk_chars": 80,

    # -----------------------------
    # Embeddings
    # -----------------------------
    "embedding_model": "intfloat/multilingual-e5-small",
    "embedding_batch_size": 64,

    # -----------------------------
    # Hybrid retrieval
    # -----------------------------
    "dense_top_k": 20,
    "sparse_top_k": 20,
    "rrf_k": 60,
    "rrf_pool_k": 18,
    "final_top_k": 5,

    # -----------------------------
    # Reranking
    # -----------------------------
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L6-v2",
    "rerank_max_length": 384,

    # -----------------------------
    # Confidence gate
    # -----------------------------
    "min_rerank_score": 0.35,
    "min_retrieval_agreement": 0.18,
    "min_final_confidence": 0.42,
    "confidence_margin_bonus": 0.10,

    # -----------------------------
    # Persistent artifacts
    # -----------------------------
    "artifact_dir": "./parkville_rag_artifacts",
    "chroma_dir": "./parkville_rag_artifacts/chroma",
    "chroma_collection": "parkville_skin_hair_expert",

    # -----------------------------
    # LLM
    # -----------------------------
    "llm_provider": "groq_openai_compatible",
    "llm_base_url": "https://api.groq.com/openai/v1",
    "llm_model": "openai/gpt-oss-120b",
    "temperature": 0.05,
    "max_tokens": 1000,
    "max_retries": 3,

    # -----------------------------
    # Evaluation
    # -----------------------------
    "eval_retrieval_k": 5,
}

Path(CONFIG["artifact_dir"]).mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")


Configuration loaded.


## 2 — Upload the two source PDFs

The runtime expects **PDFs**, not Markdown. Upload both files in one step.

The uploader also tries to infer which file is Skin Care vs Hair Care from the filename. If it cannot infer the mapping safely, set the two paths manually in the next cell.


In [6]:

# Google Colab upload cell.
# On a local Jupyter environment, skip this cell and set SKIN_CARE_PDF / HAIR_CARE_PDF manually.

try:
    from google.colab import files
    uploaded = files.upload()
    pdf_candidates = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
except ImportError:
    print("Not running in Google Colab.")
    print("Set pdf_candidates manually, e.g.: pdf_candidates = ['skin.pdf', 'hair.pdf']")
    pdf_candidates = []

if len(pdf_candidates) != 2:
    raise ValueError(f"Expected exactly 2 PDF files, found {len(pdf_candidates)}: {pdf_candidates}")

print("Uploaded PDFs:")
for p in pdf_candidates:
    print(" -", p)


Saving Skin_Care_Pregnancy_Safety_Review_Updated (2).pdf to Skin_Care_Pregnancy_Safety_Review_Updated (2) (1).pdf
Saving Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf to Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf
Uploaded PDFs:
 - Skin_Care_Pregnancy_Safety_Review_Updated (2) (1).pdf
 - Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf


In [7]:

# Automatic role assignment based on filename.
# If the filenames are ambiguous, replace these two variables with the correct paths.

def infer_pdf_role(filename: str) -> Optional[str]:
    name = filename.lower()
    if any(x in name for x in ["skin", "skincare", "skin-care", "face"]):
        return "skin_care"
    if any(x in name for x in ["hair", "haircare", "hair-care"]):
        return "hair_care"
    return None

role_map = {p: infer_pdf_role(p) for p in pdf_candidates}
print("Inferred roles:", role_map)

if set(v for v in role_map.values() if v) == {"skin_care", "hair_care"}:
    SKIN_CARE_PDF = next(p for p, role in role_map.items() if role == "skin_care")
    HAIR_CARE_PDF = next(p for p, role in role_map.items() if role == "hair_care")
else:
    print("Could not infer both roles safely.")
    print("Set SKIN_CARE_PDF and HAIR_CARE_PDF manually, e.g.:")
    print("SKIN_CARE_PDF = pdf_candidates[0]")
    print("HAIR_CARE_PDF = pdf_candidates[1]")
    raise ValueError("Ambiguous PDF roles — assign SKIN_CARE_PDF and HAIR_CARE_PDF explicitly.")

SOURCE_DOCUMENTS = [
    {
        "path": SKIN_CARE_PDF,
        "category": "skin_care",
        "document_id": "SKIN-KB-001",
        "title": "Parkville Skin Care Pregnancy Safety Review",
    },
    {
        "path": HAIR_CARE_PDF,
        "category": "hair_care",
        "document_id": "HAIR-KB-001",
        "title": "Parkville Hair Care Pregnancy Safety Review",
    },
]

print("\nRegistered sources:")
for d in SOURCE_DOCUMENTS:
    print(f"- {d['category']}: {d['path']} -> {d['document_id']}")


Inferred roles: {'Skin_Care_Pregnancy_Safety_Review_Updated (2) (1).pdf': 'skin_care', 'Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf': 'hair_care'}

Registered sources:
- skin_care: Skin_Care_Pregnancy_Safety_Review_Updated (2) (1).pdf -> SKIN-KB-001
- hair_care: Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf -> HAIR-KB-001


## 3 — PDF extraction with page-level traceability

In [8]:

import fitz  # PyMuPDF

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def normalize_pdf_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\r", "\n")
    # Join excessive spaces while preserving line structure.
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def extract_pdf_pages(path: str, document_id: str, category: str, title: str) -> List[Dict[str, Any]]:
    pages = []
    pdf = fitz.open(path)

    for page_idx in range(len(pdf)):
        raw = pdf[page_idx].get_text("text") or ""
        text = normalize_pdf_text(raw)

        pages.append({
            "document_id": document_id,
            "document_name": Path(path).name,
            "title": title,
            "category": category,
            "page_number": page_idx + 1,
            "text": text,
        })
    pdf.close()
    return pages

all_pages: List[Dict[str, Any]] = []
pdf_stats = []

for src in SOURCE_DOCUMENTS:
    pages = extract_pdf_pages(
        src["path"], src["document_id"], src["category"], src["title"]
    )
    all_pages.extend(pages)

    nonempty = [p for p in pages if len(p["text"]) >= CONFIG["min_page_chars"]]
    stats = {
        "file": src["path"],
        "pages": len(pages),
        "nonempty_pages": len(nonempty),
        "total_chars": sum(len(p["text"]) for p in pages),
        "sha256": sha256_file(src["path"]),
    }
    pdf_stats.append(stats)
    print(stats)

if not all_pages:
    raise RuntimeError("No PDF pages were extracted.")

display(pd.DataFrame(pdf_stats))


{'file': 'Skin_Care_Pregnancy_Safety_Review_Updated (2) (1).pdf', 'pages': 17, 'nonempty_pages': 17, 'total_chars': 34310, 'sha256': '0b80a0d17f1691775c8f6c2a725ddb6c7c39636f9b9511dadbbe37c17d196aaf'}
{'file': 'Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf', 'pages': 23, 'nonempty_pages': 23, 'total_chars': 37132, 'sha256': '1a925ce3fcfc4d0296f3f9de6efa1972d9d69e956b7ba33c02b280b2ae005ddc'}


,file,pages,nonempty_pages,total_chars,sha256
0,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,17,17,34310,0b80a0d17f1691775c8f6c2a725ddb6c7c39636f9b9511...
1,Hair_Care_Pregnancy_Safety_REBUILT_FINAL (2).pdf,23,23,37132,1a925ce3fcfc4d0296f3f9de6efa1972d9d69e956b7ba3...


### Extraction quality check

If a PDF is scanned/image-only, extraction will be sparse. This notebook **does not silently hallucinate OCR content**. Instead, it flags weak pages so you can add an OCR stage deliberately if needed.


In [9]:

weak_pages = [
    (p["document_name"], p["page_number"], len(p["text"]))
    for p in all_pages
    if len(p["text"]) < CONFIG["min_page_chars"]
]

print(f"Pages extracted: {len(all_pages)}")
print(f"Weak/near-empty pages: {len(weak_pages)}")

if weak_pages:
    print("\nWARNING: some pages contain very little extracted text.")
    print("First 15 weak pages:")
    for row in weak_pages[:15]:
        print("  ", row)
else:
    print("Extraction quality looks healthy.")


Pages extracted: 40
Weak/near-empty pages: 0
Extraction quality looks healthy.


## 4 — Product-aware structural parsing

In [10]:

# The source documents are product catalogs/reviews. We preserve product boundaries
# instead of blindly splitting every N characters.

PRODUCT_PATTERNS = [
    # "### 1. Product Name"
    re.compile(r"^\s*#{2,6}\s*(\d{1,3})\.\s*(.+?)\s*$"),
    # "1. Product Name"
    re.compile(r"^\s*(\d{1,3})\.\s+([A-Z][^\n]{2,120})\s*$"),
]

SECTION_PATTERNS = [
    re.compile(r"^\s*#{2,6}\s*(.+?)\s*$"),
    re.compile(r"^\s*\*\*(.+?)\*\*\s*$"),
    re.compile(r"^\s*(Overview|Ingredients|Benefits|Key Benefits|How to Use|Ideal For|"
               r"Product Details|Pregnancy Safety|Skin Type|Hair Type|Usage|Frequency|"
               r"Sun Exposure(?: Safety| safety)?)\s*:?\s*$", re.I),
]

def clean_markup(s: str) -> str:
    s = re.sub(r"\*{1,3}", "", s)
    s = re.sub(r"`", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip(" -:#\t")

def detect_product_heading(line: str) -> Optional[Tuple[str, str]]:
    clean = clean_markup(line)
    for pat in PRODUCT_PATTERNS:
        m = pat.match(clean)
        if m:
            number = m.group(1)
            name = clean_markup(m.group(2))
            # Prevent generic headings from being treated as products.
            if len(name) >= 3 and not name.lower().startswith(("how to", "source", "contents")):
                return number, name
    return None

def detect_section_heading(line: str) -> Optional[str]:
    clean = clean_markup(line)
    if not clean:
        return None
    for pat in SECTION_PATTERNS:
        m = pat.match(clean)
        if m:
            label = clean_markup(m.group(1) if m.lastindex else clean)
            return label
    # Keyword-based fallback for extracted PDFs where markdown symbols disappear.
    low = clean.lower().rstrip(":")
    for key in [
        "overview", "ingredients", "benefits", "key benefits", "how to use",
        "ideal for", "product details", "pregnancy safety", "skin type", "hair type",
        "usage", "frequency", "sun exposure", "sun exposure safety"
    ]:
        if low == key:
            return key.title()
    return None

def parse_products(pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records = []

    for page in pages:
        lines = [ln.strip() for ln in page["text"].splitlines() if ln.strip()]
        current_product = None
        current_section = "General"
        buffer = []

        def flush():
            nonlocal buffer, current_product, current_section
            if not buffer:
                return
            text = "\n".join(buffer).strip()
            if text:
                records.append({
                    **{k: page[k] for k in [
                        "document_id", "document_name", "title", "category", "page_number"
                    ]},
                    "product_number": current_product[0] if current_product else None,
                    "product": current_product[1] if current_product else None,
                    "section": current_section,
                    "text": text,
                })
            buffer = []

        for line in lines:
            product_hit = detect_product_heading(line)
            if product_hit:
                flush()
                current_product = product_hit
                current_section = "Product Overview"
                buffer = [clean_markup(line)]
                continue

            section_hit = detect_section_heading(line)
            if section_hit:
                flush()
                current_section = section_hit
                buffer = [clean_markup(line)]
                continue

            buffer.append(line)

        flush()

    return records

sections = parse_products(all_pages)

df_sections = pd.DataFrame(sections)
print("Structured sections:", len(df_sections))
display(df_sections.head(15)[[
    "category", "document_name", "page_number", "product_number", "product", "section", "text"
]])


Structured sections: 110


,category,document_name,page_number,product_number,product,section,text
0,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,1,None,None,General,Parkville Skin Care – Pregnancy Safety Review\...
1,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,1,1,Shaan Cleanser,Product Overview,1. Shaan Cleanser\nOverview: Gentle Hydrating ...
2,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,1,2,Shaan Rejuvenation Cream,Product Overview,2. Shaan Rejuvenation Cream\nShaan Rejuvenatio...
3,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,2,None,None,General,•\nVitamin E\n•\nHoney\n•\n5 Natural Oils\n•\n...
4,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,2,3,Shaan Soothing Gel,Product Overview,3. Shaan Soothing Gel\nShaan Soothing Gel Over...
5,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,2,4,"Revitalizing Eye Cream – Brighten, Hydrate & R...",Product Overview,"4. Revitalizing Eye Cream – Brighten, Hydrate ..."
6,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,3,None,None,General,•\nGlabridin\n•\nCoffee Arabica Seed Extract\n...
7,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,3,5,Shaan Body Cream – Intensive Hydration & Barri...,Product Overview,5. Shaan Body Cream – Intensive Hydration & Ba...
8,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,3,6,Shaan Hydrating Body Shower Creamy Cleanser – ...,Product Overview,6. Shaan Hydrating Body Shower Creamy Cleanser...
9,skin_care,Skin_Care_Pregnancy_Safety_Review_Updated (2) ...,4,None,None,General,"•\nSuitable for All Skin Types\n•\nCreamy, Nou..."


### Structural parser fallback

Some PDF exports lose heading formatting. When the product parser cannot detect enough product boundaries, the system automatically falls back to **page-aware chunks** instead of pretending that the PDF structure was recovered.


In [11]:

def build_page_fallback(pages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    fallback = []
    for p in pages:
        if not p["text"]:
            continue
        fallback.append({
            **{k: p[k] for k in ["document_id", "document_name", "title", "category", "page_number"]},
            "product_number": None,
            "product": None,
            "section": "Page",
            "text": p["text"],
        })
    return fallback

products_detected = int(df_sections["product"].notna().sum()) if not df_sections.empty else 0
unique_products = df_sections.loc[df_sections["product"].notna(), ["category", "product"]].drop_duplicates()

print("Detected product-bearing sections:", products_detected)
print("Detected unique products:", len(unique_products))
display(unique_products.head(30))

if len(unique_products) < 10:
    print("\nWARNING: Product heading detection appears weak. Using page-aware fallback.")
    sections = build_page_fallback(all_pages)
else:
    print("\nProduct-aware structure accepted.")


Detected product-bearing sections: 70
Detected unique products: 61


,category,product
1,skin_care,Shaan Cleanser
2,skin_care,Shaan Rejuvenation Cream
4,skin_care,Shaan Soothing Gel
5,skin_care,"Revitalizing Eye Cream – Brighten, Hydrate & R..."
7,skin_care,Shaan Body Cream – Intensive Hydration & Barri...
8,skin_care,Shaan Hydrating Body Shower Creamy Cleanser – ...
10,skin_care,SHAAN Dry S Cream – Fragrance-Free Care for Dr...
11,skin_care,Shaan Cica Healing Moisturizer – Soothing & Sk...
13,skin_care,SHAAN Hydrating Therma Micellar Water – Gentle...
14,skin_care,Shaan Body Milk – Vanilla Coconut | 72-Hour De...



Product-aware structure accepted.


## 5 — Product-aware chunking + metadata enrichment

In [12]:

def split_long_text(text: str, target: int, overlap: int, max_len: int) -> List[str]:
    text = text.strip()
    if len(text) <= max_len:
        return [text]

    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + target, n)

        # Prefer a natural boundary near the target.
        if end < n:
            candidates = [
                text.rfind("\n\n", start, end),
                text.rfind("\n", start, end),
                text.rfind(". ", start, end),
                text.rfind("; ", start, end),
                text.rfind(" ", start, end),
            ]
            best = max(candidates)
            if best > start + int(target * 0.55):
                end = best + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        next_start = max(end - overlap, start + 1)
        if next_start >= n:
            break
        start = next_start

    return chunks

SAFETY_LABELS = [
    "AVOID DURING PREGNANCY",
    "CONSULT A DOCTOR FIRST",
    "SAFE",
]

def infer_safety_label(text: str) -> Optional[str]:
    upper = text.upper()
    # Check the longest phrases first.
    if "AVOID DURING PREGNANCY" in upper:
        return "AVOID DURING PREGNANCY"
    if "CONSULT A DOCTOR FIRST" in upper:
        return "CONSULT A DOCTOR FIRST"
    # Avoid treating every occurrence of the word SAFE as a safety label.
    if re.search(r"(PREGNANCY\s+SAFETY\s*[:\-]?\s*SAFE|PREGNANCY\s+SAFETY:\s*SAFE|\bPREGNANCY\s+SAFETY\b.*\bSAFE\b)", upper):
        return "SAFE"
    return None

def infer_section_type(section: str) -> str:
    s = section.lower()
    if "pregnancy" in s:
        return "pregnancy_safety"
    if "ingredient" in s:
        return "ingredients"
    if "benefit" in s:
        return "benefits"
    if "use" in s or "frequency" in s or "usage" in s:
        return "usage"
    if "skin type" in s:
        return "skin_type"
    if "hair type" in s:
        return "hair_type"
    if "overview" in s or "general" in s or "page" in s:
        return "overview"
    return "other"

def make_chunks(section_records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks = []
    counters = {}

    for rec in section_records:
        base_id = rec["document_id"]
        counters.setdefault(base_id, 0)

        pieces = split_long_text(
            rec["text"],
            target=CONFIG["chunk_target_chars"],
            overlap=CONFIG["chunk_overlap_chars"],
            max_len=CONFIG["max_chunk_chars"],
        )

        for piece_idx, piece in enumerate(pieces, start=1):
            if len(piece) < CONFIG["min_chunk_chars"]:
                continue

            counters[base_id] += 1
            chunk_id = f"{base_id}-CH-{counters[base_id]:04d}"

            safety = infer_safety_label(piece)
            section_type = infer_section_type(rec["section"])

            # Lightweight brand extraction: first token when a product name exists.
            product = rec.get("product")
            brand = None
            if product:
                brand = re.split(r"\s+", product.strip())[0]

            metadata = {
                "chunk_id": chunk_id,
                "document_id": rec["document_id"],
                "document_name": rec["document_name"],
                "title": rec["title"],
                "category": rec["category"],
                "page_number": int(rec["page_number"]),
                "product_number": str(rec["product_number"]) if rec.get("product_number") is not None else "",
                "product": product or "",
                "brand": brand or "",
                "section": rec["section"],
                "section_type": section_type,
                "piece_index": piece_idx,
                "safety_label": safety or "",
            }

            chunks.append({
                "chunk_id": chunk_id,
                "text": piece,
                "metadata": metadata,
            })

    return chunks

chunks = make_chunks(sections)

if not chunks:
    raise RuntimeError("No chunks were produced.")

chunk_df = pd.DataFrame([
    {**c["metadata"], "text_chars": len(c["text"]), "text": c["text"]}
    for c in chunks
])

print("Total chunks:", len(chunks))
display(
    chunk_df.groupby(["category", "section_type"]).size().reset_index(name="chunks")
)
display(chunk_df.head(10)[[
    "chunk_id", "category", "brand", "product", "section", "page_number", "safety_label", "text_chars"
]])

ALL_PRODUCTS = sorted(
    {c["metadata"]["product"].strip() for c in chunks if c["metadata"]["product"].strip()},
    key=len, reverse=True
)
ALL_BRANDS = sorted(
    {c["metadata"]["brand"].strip() for c in chunks if c["metadata"]["brand"].strip()},
    key=len, reverse=True
)
print("Unique products indexed:", len(ALL_PRODUCTS))
print("Unique brands indexed:", len(ALL_BRANDS))


Total chunks: 303


,category,section_type,chunks
0,hair_care,benefits,1
1,hair_care,other,1
2,hair_care,overview,252
3,hair_care,usage,1
4,skin_care,benefits,2
5,skin_care,ingredients,2
6,skin_care,overview,44


,chunk_id,category,brand,product,section,page_number,safety_label,text_chars
0,SKIN-KB-001-CH-0001,skin_care,,,General,1,CONSULT A DOCTOR FIRST,423
1,SKIN-KB-001-CH-0002,skin_care,Shaan,Shaan Cleanser,Product Overview,1,SAFE,1039
2,SKIN-KB-001-CH-0003,skin_care,Shaan,Shaan Rejuvenation Cream,Product Overview,1,,760
3,SKIN-KB-001-CH-0004,skin_care,,,General,2,SAFE,470
4,SKIN-KB-001-CH-0005,skin_care,Shaan,Shaan Soothing Gel,Product Overview,2,SAFE,1222
5,SKIN-KB-001-CH-0006,skin_care,Revitalizing,"Revitalizing Eye Cream – Brighten, Hydrate & R...",Product Overview,2,,448
6,SKIN-KB-001-CH-0007,skin_care,,,General,3,SAFE,518
7,SKIN-KB-001-CH-0008,skin_care,Shaan,Shaan Body Cream – Intensive Hydration & Barri...,Product Overview,3,SAFE,1215
8,SKIN-KB-001-CH-0009,skin_care,Shaan,Shaan Hydrating Body Shower Creamy Cleanser – ...,Product Overview,3,,469
9,SKIN-KB-001-CH-0010,skin_care,,,General,4,SAFE,696


Unique products indexed: 59
Unique brands indexed: 15


## 6 — Dense embedding index (Chroma)

The dense retriever handles semantic paraphrases. The index is persisted to disk and is fully rebuildable from the uploaded PDFs.

We intentionally embed the **product + section + chunk text** together. This helps a query like “pregnancy safety of the cleanser” remain anchored to the correct product/section.


In [13]:

from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer(CONFIG["embedding_model"])

def embedding_text(c: Dict[str, Any]) -> str:
    m = c["metadata"]
    prefix = " | ".join([
        f"category={m['category']}",
        f"brand={m['brand']}",
        f"product={m['product']}",
        f"section={m['section']}",
    ])
    return f"{prefix}\n{c['text']}"

embed_inputs = [embedding_text(c) for c in chunks]

embeddings = embedding_model.encode(
    embed_inputs,
    batch_size=CONFIG["embedding_batch_size"],
    show_progress_bar=True,
    normalize_embeddings=True,
).astype("float32")

print("Embedding matrix:", embeddings.shape)

chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_dir"])

# Clean rebuild to avoid duplicate chunks when rerunning ingestion.
try:
    chroma_client.delete_collection(CONFIG["chroma_collection"])
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name=CONFIG["chroma_collection"],
    metadata={"hnsw:space": "cosine"},
)

metadatas = [c["metadata"] for c in chunks]
documents = [c["text"] for c in chunks]
ids = [c["chunk_id"] for c in chunks]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatas,
)

print("Chroma collection count:", collection.count())


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding matrix: (303, 384)
Chroma collection count: 303


## 7 — Sparse BM25 index

In [14]:

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> List[str]:
      # Preserve numbers and ingredient/product tokens; normalize punctuation.

    return re.findall(r"[a-z0-9\u0600-\u06FF][a-z0-9\u0600-\u06FF%+.-]*", text.lower())

bm25_tokens = [tokenize(embedding_text(c)) for c in chunks]
bm25 = BM25Okapi(bm25_tokens)

print("BM25 corpus size:", len(bm25_tokens))


BM25 corpus size: 303


## 8 — Query understanding

We classify the question into an internal retrieval intent. This is **routing metadata**, not a source of truth.

Examples:
- `pregnancy_safety`
- `ingredients`
- `usage`
- `benefits`
- `comparison`
- `product_lookup`
- `out_of_scope`


In [15]:
from rapidfuzz import fuzz

INTENT_KEYWORDS = {
    "pregnancy_safety": [
        "pregnant", "pregnancy", "expecting", "breastfeeding", "safe during pregnancy",
        "avoid during pregnancy", "doctor first", "pregnancy safety",
    ],
    "ingredients": ["ingredient", "ingredients", "contains", "formula", "inci"],
    "usage": ["how to use", "use it", "apply", "frequency", "how often", "routine"],
    "benefits": ["benefit", "benefits", "helps", "good for", "what does it do"],
    "skin_type": ["skin type", "dry skin", "oily skin", "combination skin", "sensitive skin"],
    "hair_type": ["hair type", "dry hair", "oily scalp", "frizzy hair", "damaged hair"],
    "comparison": ["compare", "comparison", "difference", "versus", "vs"],
}

OUT_OF_SCOPE_PATTERNS = [
    "diagnose", "diagnosis", "what disease do i have", "prescribe", "prescription",
    "what medication should i take", "blood test", "medical diagnosis",
    "dose of", "dosage of", "emergency", "hospital", "doctor for my condition",
]

def detect_category(query: str) -> Optional[str]:
    q = query.lower()
    skin_terms = ["skin", "face", "cleanser", "cream", "serum", "sunscreen", "toner", "lip balm"]
    hair_terms = ["hair", "shampoo", "conditioner", "scalp", "hair fall", "hair loss", "hair mask", "hair serum"]
    s = sum(t in q for t in skin_terms)
    h = sum(t in q for t in hair_terms)
    if s > h and s > 0:
        return "skin_care"
    if h > s and h > 0:
        return "hair_care"
    return None

def detect_intent(query: str) -> str:
    q = query.lower().strip()
    if any(p in q for p in OUT_OF_SCOPE_PATTERNS):
        return "out_of_scope"
    scores = {
        intent: sum(1 for kw in kws if kw in q)
        for intent, kws in INTENT_KEYWORDS.items()
    }
    best = max(scores, key=scores.get)
    if scores[best] > 0:
        return best
    return "product_lookup"



def extract_product_hints(query: str, fuzzy_threshold: int = 85) -> List[str]:
    q = query.lower()
    exact = [p for p in ALL_PRODUCTS if p.lower() in q]
    if exact:
        return sorted(set(exact), key=len, reverse=True)

    # Fallback: fuzzy match لو مفيش exact match (بيلقط الأخطاء الإملائية)
    fuzzy_hits = [
        p for p in ALL_PRODUCTS
        if fuzz.partial_ratio(p.lower(), q) >= fuzzy_threshold
    ]
    return sorted(set(fuzzy_hits), key=len, reverse=True)

def extract_brand_hints(query: str) -> List[str]:
    q = query.lower()
    return [b for b in ALL_BRANDS if b.lower() in q]

def analyze_query(query: str) -> Dict[str, Any]:
    return {
        "category": detect_category(query),
        "intent": detect_intent(query),
        "products": extract_product_hints(query),
        "brands": extract_brand_hints(query),
    }

for q in [
    "Is CLARY Leave-In Cream safe during pregnancy?",
    "What are the ingredients in Shaan Cleanser?",
    "Compare Clary Hair Mask and Seropipe Hair Mask.",
]:
    print(q, "->", analyze_query(q))


Is CLARY Leave-In Cream safe during pregnancy? -> {'category': 'skin_care', 'intent': 'pregnancy_safety', 'products': [], 'brands': ['CLARY', 'Clary']}
What are the ingredients in Shaan Cleanser? -> {'category': 'skin_care', 'intent': 'ingredients', 'products': ['Shaan Cleanser'], 'brands': ['Shaan', 'SHAAN']}
Compare Clary Hair Mask and Seropipe Hair Mask. -> {'category': 'hair_care', 'intent': 'comparison', 'products': [], 'brands': ['Seropipe', 'CLARY', 'Clary']}


## 9 — Hybrid retrieval + RRF

In [16]:

def metadata_match(meta: Dict[str, Any], analysis: Dict[str, Any]) -> bool:
    category = analysis.get("category")
    if category and meta.get("category") != category:
        return False

    products = analysis.get("products") or []
    if products and not any(meta.get("product", "").lower() == p.lower() for p in products):
        # Keep comparison/product-free retrieval broad; exact product hint is a strong filter.
        return False

    brands = analysis.get("brands") or []
    if brands and not any(meta.get("brand", "").lower() == b.lower() for b in brands):
        return False

    return True

def dense_search(query: str, k: int, analysis: Dict[str, Any]) -> List[Tuple[str, Dict[str, Any], float]]:
    q_emb = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    ).astype("float32")[0].tolist()

    # Pull a slightly larger pool, then apply metadata filtering client-side.
    raw = collection.query(
        query_embeddings=[q_emb],
        n_results=min(max(k * 4, k), len(chunks)),
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for doc_text, meta, distance in zip(
        raw["documents"][0],
        raw["metadatas"][0],
        raw["distances"][0],
    ):
        if not metadata_match(meta, analysis):
            continue
        # cosine distance -> approximate relevance
        score = 1.0 - float(distance)
        hits.append((meta["chunk_id"], {"text": doc_text, "metadata": meta}, score))
        if len(hits) >= k:
            break
    return hits

def sparse_search(query: str, k: int, analysis: Dict[str, Any]) -> List[Tuple[str, Dict[str, Any], float]]:
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    ranked_idx = np.argsort(scores)[::-1]

    hits = []
    for idx in ranked_idx:
        c = chunks[int(idx)]
        if not metadata_match(c["metadata"], analysis):
            continue
        hits.append((
            c["chunk_id"],
            {"text": c["text"], "metadata": c["metadata"]},
            float(scores[int(idx)]),
        ))
        if len(hits) >= k:
            break
    return hits

def rrf_fusion(*ranked_lists, rrf_k: int = 60) -> List[Tuple[str, Dict[str, Any], float]]:
    fused_score = {}
    doc_lookup = {}

    for ranked_list in ranked_lists:
        for rank, (chunk_id, doc, _) in enumerate(ranked_list, start=1):
            fused_score[chunk_id] = fused_score.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
            doc_lookup[chunk_id] = doc

    ordered = sorted(fused_score.items(), key=lambda x: x[1], reverse=True)
    return [(cid, doc_lookup[cid], score) for cid, score in ordered]

print("Hybrid retrieval functions ready.")


Hybrid retrieval functions ready.


## 10 — Cross-Encoder reranking

In [17]:

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    CONFIG["reranker_model"],
    max_length=CONFIG["rerank_max_length"],
)

def rerank_results(query: str, fused_results, top_n: int, pool_k: int) -> List[Tuple[str, Dict[str, Any], float]]:
    if not fused_results:
        return []

    candidates = fused_results[:pool_k]
    pairs = [[query, item[1]["text"]] for item in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)

    reranked = []
    for (chunk_id, doc, _rrf_score), score in zip(candidates, scores):
        reranked.append((chunk_id, doc, float(score)))

    reranked.sort(key=lambda x: x[2], reverse=True)
    return reranked[:top_n]

def hybrid_retrieve(query: str) -> Dict[str, Any]:
    analysis = analyze_query(query)

    if analysis["intent"] == "out_of_scope":
        return {
            "analysis": analysis,
            "dense": [],
            "sparse": [],
            "fused": [],
            "reranked": [],
        }

    # Set/list questions need broader evidence gathering than a single-product lookup.
    qlow = query.lower()
    is_set_query = (
        analysis["intent"] == "pregnancy_safety"
        and any(term in qlow for term in ["which products", "what products", "list the products"])
    )

    dense_k = 40 if is_set_query else CONFIG["dense_top_k"]
    sparse_k = 40 if is_set_query else CONFIG["sparse_top_k"]
    final_k = min(12, len(chunks)) if is_set_query else CONFIG["final_top_k"]

    dense_hits = dense_search(query, dense_k, analysis)
    sparse_hits = sparse_search(query, sparse_k, analysis)
    fused = rrf_fusion(dense_hits, sparse_hits, rrf_k=CONFIG["rrf_k"])

    rerank_pool = min(40 if is_set_query else CONFIG["rrf_pool_k"], len(fused))
    reranked = rerank_results(query, fused, final_k, pool_k=rerank_pool)

    return {
        "analysis": analysis,
        "dense": dense_hits,
        "sparse": sparse_hits,
        "fused": fused,
        "reranked": reranked,
    }

# Smoke test — retrieval only.
demo = hybrid_retrieve("Which products should be avoided during pregnancy?")
print("Intent:", demo["analysis"])
for rank, (cid, doc, score) in enumerate(demo["reranked"], 1):
    m = doc["metadata"]
    print(f"{rank}. {score:.4f} | {m['category']} | {m['product']} | {m['section']} | p.{m['page_number']}")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Intent: {'category': None, 'intent': 'pregnancy_safety', 'products': [], 'brands': ['Product']}
1. -11.1338 | hair_care | Product descriptions and ingredient lists supplied in the conversation and in the original Hair Care review file. | Product Overview | p.23


## 11 — Confidence scoring & refusal gate

A professional RAG system should **not answer simply because the LLM can produce a fluent sentence**.

The confidence gate combines:

- Cross-encoder relevance
- Dense/BM25 agreement
- Top-vs-second reranker margin
- Exact product/brand signal
- Safety-section availability for pregnancy questions

Low-confidence queries are refused before generation.


In [18]:

def chunk_id_set(items):
    return {x[0] for x in items}

def compute_confidence(retrieval: Dict[str, Any]) -> Dict[str, Any]:
    analysis = retrieval["analysis"]
    reranked = retrieval["reranked"]
    dense_ids = chunk_id_set(retrieval["dense"])
    sparse_ids = chunk_id_set(retrieval["sparse"])

    if not reranked:
        return {"score": 0.0, "allowed": False, "reason": "no_retrieval"}

    top_score = float(reranked[0][2])
    second_score = float(reranked[1][2]) if len(reranked) > 1 else top_score - 0.1
    margin = max(0.0, top_score - second_score)

    # Approximate agreement among the two retrieval channels.
    union = len(dense_ids | sparse_ids) or 1
    intersection = len(dense_ids & sparse_ids)
    agreement = intersection / union

    exact_product_signal = 0.0
    if analysis.get("products"):
        exact_product_signal = 1.0

    safety_signal = 0.0
    if analysis["intent"] == "pregnancy_safety":
        safety_hits = [
            x for x in reranked
            if x[1]["metadata"].get("safety_label")
            and x[1]["metadata"].get("section_type") == "pregnancy_safety"
        ]
        safety_signal = 1.0 if safety_hits else 0.0

    # Normalize the cross-encoder score heuristically into [0, 1].
    rerank_component = 1.0 / (1.0 + math.exp(-top_score))

    score = (
        0.55 * rerank_component
        + 0.20 * agreement
        + 0.10 * min(margin * 2.0, 1.0)
        + 0.10 * exact_product_signal
        + 0.05 * safety_signal
    )

    allowed = (
    rerank_component >= CONFIG["min_rerank_score"]
    and agreement >= CONFIG["min_retrieval_agreement"]
    and score >= CONFIG["min_final_confidence"]
)

    return {
    "score": round(float(score), 4),
    "allowed": bool(allowed),
    "top_rerank_score": round(top_score, 4),
    "top_rerank_normalized": round(rerank_component, 4),
    "margin": round(margin, 4),
    "agreement": round(agreement, 4),
    "reason": "pass" if allowed else "low_confidence",
}


## 12 — LLM client

In [19]:

from getpass import getpass
from openai import OpenAI

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY (hidden): ")

llm_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url=CONFIG["llm_base_url"],
    timeout=30.0,
    max_retries=0,
)


print("LLM client ready:", CONFIG["llm_model"])


Enter GROQ_API_KEY (hidden): ··········
LLM client ready: openai/gpt-oss-120b


## 13 — Grounded response schema + system prompt

In [21]:

RESPONSE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "status": {
            "type": "string",
            "enum": ["answered", "insufficient_evidence", "out_of_scope"]
        },
        "answer": {"type": "string"},
        "evidence_summary": {"type": "string"},
        "citations": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "chunk_id": {"type": "string"},
                    "document": {"type": "string"},
                    "page": {"type": "integer"},
                    "section": {"type": "string"},
                    "product": {"type": "string"},
                },
                "required": ["chunk_id", "document", "page", "section", "product"],
            },
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low", "insufficient"]
        },
    },
    "required": ["status", "answer", "evidence_summary", "citations", "confidence"],
}

SYSTEM_PROMPT = """You are the Parkville Skin & Hair Care Expert.

You are a strict, citation-bound RAG assistant.
Your ONLY source of truth is the retrieved context supplied in the user message.

Rules:
1. Never use outside medical, cosmetic, product, or ingredient knowledge.
2. Never invent a product, ingredient, benefit, usage instruction, pregnancy label, brand, or citation.
3. Preserve pregnancy-safety labels EXACTLY when they appear in the context:
   SAFE
   CONSULT A DOCTOR FIRST
   AVOID DURING PREGNANCY
4. Do not infer a pregnancy label from an ingredient unless that classification is explicitly present in the retrieved source.
5. Every substantive answer must be supported by one or more cited chunk_ids.
6. If the context is insufficient, say so explicitly and return status="insufficient_evidence".
7. If the question is outside the supported product-information domain, return status="out_of_scope".
8. Keep answers concise but useful. Mention uncertainty when the source is incomplete.
9. Return ONLY valid JSON matching the supplied schema. No markdown fences.

You can answer questions about:
- Product names and descriptions
- Ingredients
- Key benefits
- Skin or hair type
- How to use
- Usage frequency
- Product properties
- Sun exposure information
- Pregnancy safety
- Comparisons between products, when the supplied context contains information about both products
"""


## 14 — Generation, parsing, and retries

In [22]:

def build_context(reranked: List[Tuple[str, Dict[str, Any], float]]) -> str:
    blocks = []
    for rank, (chunk_id, doc, score) in enumerate(reranked, 1):
        m = doc["metadata"]
        blocks.append(
            f"[SOURCE {rank}]\n"
            f"chunk_id: {chunk_id}\n"
            f"document: {m['document_name']}\n"
            f"page: {m['page_number']}\n"
            f"category: {m['category']}\n"
            f"brand: {m['brand']}\n"
            f"product: {m['product']}\n"
            f"section: {m['section']}\n"
            f"safety_label: {m['safety_label']}\n"
            f"rerank_score: {score:.4f}\n"
            f"text:\n{doc['text']}\n"
        )
    return "\n\n".join(blocks)

def extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    # First try direct JSON.
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Then extract the outermost JSON object.
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError("No JSON object found in model output.")

@retry(
    stop=stop_after_attempt(CONFIG["max_retries"]),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type(Exception),
    reraise=True,
)
def call_llm(messages):
    response = llm_client.chat.completions.create(
        model=CONFIG["llm_model"],
        messages=messages,
        temperature=CONFIG["temperature"],
        max_tokens=CONFIG["max_tokens"],
    )
    return response.choices[0].message.content

def validate_response(obj: Dict[str, Any]) -> None:
    from jsonschema import validate
    validate(instance=obj, schema=RESPONSE_SCHEMA)


## 15 — Hard guardrails

In [23]:

def validate_citations(answer: Dict[str, Any], reranked: List[Tuple[str, Dict[str, Any], float]]) -> Dict[str, Any]:
    allowed = {cid: doc["metadata"] for cid, doc, _ in reranked}
    invalid = []

    for c in answer.get("citations", []):
        cid = c.get("chunk_id")
        if cid not in allowed:
            invalid.append(cid)
            continue

        source = allowed[cid]
        if str(c.get("document")) != str(source.get("document_name")):
            invalid.append(cid)
        elif int(c.get("page")) != int(source.get("page_number")):
            invalid.append(cid)
        elif str(c.get("product", "")).strip().lower() != str(source.get("product", "")).strip().lower():
            invalid.append(cid)
        elif str(c.get("section", "")).strip().lower() != str(source.get("section", "")).strip().lower():
            invalid.append(cid)

    return {
        "valid": len(invalid) == 0,
        "invalid_chunk_ids": invalid,
        "citation_count": len(answer.get("citations", [])),
    }

def validate_safety_label(answer: Dict[str, Any], reranked: List[Tuple[str, Dict[str, Any], float]], intent: str) -> Dict[str, Any]:
    if intent != "pregnancy_safety":
        return {"valid": True, "expected_labels": [], "mentioned_labels": []}

    source_labels = {
        x[1]["metadata"].get("safety_label")
        for x in reranked
        if x[1]["metadata"].get("safety_label")
    }

    mentioned = {
        label for label in SAFETY_LABELS
        if label.lower() in answer.get("answer", "").lower()
    }

    # Safety answers must not invent a label absent from the retrieved evidence.
    invalid_mentions = mentioned - source_labels

    # If the system answers a product-level pregnancy question, it should cite evidence.
    valid = len(invalid_mentions) == 0

    return {
        "valid": valid,
        "expected_labels": sorted(source_labels),
        "mentioned_labels": sorted(mentioned),
        "invalid_mentions": sorted(invalid_mentions),
    }

def post_validate(answer: Dict[str, Any], retrieval: Dict[str, Any], confidence: Dict[str, Any]) -> Dict[str, Any]:
    validate_response(answer)

    citation_check = validate_citations(answer, retrieval["reranked"])
    safety_check = validate_safety_label(
        answer,
        retrieval["reranked"],
        retrieval["analysis"]["intent"],
    )

    if answer["status"] == "answered":
        if not citation_check["valid"]:
            raise ValueError(f"Citation guard failed: {citation_check}")
        if not safety_check["valid"]:
            raise ValueError(f"Safety guard failed: {safety_check}")

    answer["_guards"] = {
        "citation": citation_check,
        "safety": safety_check,
        "confidence_gate": confidence,
    }
    return answer


## 16 — End-to-end `ask()`

In [24]:

def refusal_response(status: str, reason: str, confidence: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    return {
        "status": status,
        "answer": (
            "I don't have enough evidence in the supplied Parkville knowledge base to answer that safely."
            if status == "insufficient_evidence"
            else
            "That question is outside the supported Parkville Skin & Hair Care product-information domain."
        ),
        "evidence_summary": reason,
        "citations": [],
        "confidence": "insufficient",
        "_guards": {
            "confidence_gate": confidence or {},
        }
    }

def log_interaction(question: str, result: Dict[str, Any]) -> None:
    log_path = Path(CONFIG["artifact_dir"]) / "query_log.jsonl"
    safe_result = {k: v for k, v in result.items() if k != "_retrieval"}
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(
            {"ts": time.strftime("%Y-%m-%d %H:%M:%S"), "question": question, **safe_result},
            ensure_ascii=False, default=str
        ) + "\n")

def ask(question: str, debug: bool = False) -> Dict[str, Any]:
    started = time.perf_counter()

    q = (question or "").strip()
    if not q:
        result = refusal_response("insufficient_evidence", "Empty question.")
        log_interaction(q, result)
        return result
    if len(q) > CONFIG["max_question_chars"]:
        result = refusal_response("insufficient_evidence", "Question exceeds the configured length limit.")
        log_interaction(q, result)
        return result

    retrieval = hybrid_retrieve(q)
    analysis = retrieval["analysis"]

    if analysis["intent"] == "out_of_scope":
        result = refusal_response("out_of_scope", "Scope guard classified the query as outside product information.")
        result["_trace"] = {
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "analysis": analysis,
        }
        log_interaction(q, result)
        return result

    confidence = compute_confidence(retrieval)

    if not confidence["allowed"]:
        result = refusal_response(
            "insufficient_evidence",
            f"Retrieval confidence below threshold: {confidence}",
            confidence,
        )
        result["_trace"] = {
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "analysis": analysis,
        }
        log_interaction(q, result)
        return result

    context = build_context(retrieval["reranked"])

    user_prompt = f"""Question:
{q}

Internal retrieval analysis:
{json.dumps(analysis, ensure_ascii=False)}

Retrieved evidence:
{context}

Return ONLY JSON matching this schema:
{json.dumps(RESPONSE_SCHEMA, ensure_ascii=False)}
"""

    raw = call_llm([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])

    try:
        answer = extract_json_object(raw)
        answer = post_validate(answer, retrieval, confidence)
    except Exception as first_error:
        try:
            repair_prompt = f"""The previous response failed validation.

Validation error:
{str(first_error)}

Previous model output:
{raw}

Return ONLY corrected JSON matching this exact schema:
{json.dumps(RESPONSE_SCHEMA, ensure_ascii=False)}
"""
            repaired = call_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": repair_prompt},
            ])
            answer = extract_json_object(repaired)
            answer = post_validate(answer, retrieval, confidence)
        except Exception as second_error:
            result = refusal_response(
                "insufficient_evidence",
                f"Generation failed validation twice. First: {first_error} | Second: {second_error}",
                confidence,
            )
            result["_trace"] = {
                "latency_ms": round((time.perf_counter() - started) * 1000, 2),
                "analysis": analysis,
                "confidence": confidence,
            }
            log_interaction(q, result)
            return result

    answer["_trace"] = {
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
        "analysis": analysis,
        "confidence": confidence,
        "dense_candidates": len(retrieval["dense"]),
        "sparse_candidates": len(retrieval["sparse"]),
        "fused_candidates": len(retrieval["fused"]),
        "final_context_chunks": len(retrieval["reranked"]),
    }

    log_interaction(q, answer)

    if debug:
        answer["_retrieval"] = retrieval
    return answer


## 17 — Example queries

In [25]:

demo_questions = [
    "What are the ingredients in Shaan Cleanser?",
    "Is CLARY Leave-In Cream safe during pregnancy?",
    "How should Clary Booster Shot be used?",
    "What is Seropipe Hair Dropper designed to help with?",
    "What blood tests should be ordered during pregnancy?",
]

for q in demo_questions:
    print("\n" + "=" * 100)
    print("Q:", q)
    try:
        result = ask(q)
        print(json.dumps(result, ensure_ascii=False, indent=2))
    except Exception as e:
        print("Pipeline error:", repr(e))



Q: What are the ingredients in Shaan Cleanser?
{
  "status": "answered",
  "answer": "The Shaan Cleanser contains the following ingredients: Vit C, Glycerin, Honey, Olive Oil, Hyaluronic Acid, Deacyl Glucoside, and Citric Acid.",
  "evidence_summary": "The product overview for Shaan Cleanser lists its ingredients as Vit C, Glycerin, Honey, Olive Oil, Hyaluronic Acid, Deacyl Glucoside, and Citric Acid.",
  "citations": [
    {
      "chunk_id": "SKIN-KB-001-CH-0002",
      "document": "Skin_Care_Pregnancy_Safety_Review_Updated (2) (1).pdf",
      "page": 1,
      "section": "Product Overview",
      "product": "Shaan Cleanser"
    }
  ],
  "confidence": "high",
  "_guards": {
    "citation": {
      "valid": true,
      "invalid_chunk_ids": [],
      "citation_count": 1
    },
    "safety": {
      "valid": true,
      "expected_labels": [],
      "mentioned_labels": []
    },
    "confidence_gate": {
      "score": 0.8699,
      "allowed": true,
      "top_rerank_score": 8.3317,
     

## 18 — 20-question Golden Evaluation Set

The evaluation set is deliberately mixed:

- Skin Care
- Hair Care
- Pregnancy Safety
- Ingredients / Usage
- Comparison
- Out-of-scope refusal

The expected references are **source anchors**, not invented medical facts.


In [26]:

GOLDEN_DATASET = [
    {
        "id": "Q01",
        "question": "What is the pregnancy safety classification of Shaan Cleanser?",
        "category": "skin_care", "intent": "pregnancy_safety",
        "expected_products": ["Shaan Cleanser"],
        "expected_labels": ["SAFE"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q02",
        "question": "What are the key benefits of Shaan Rejuvenation Cream?",
        "category": "skin_care", "intent": "benefits",
        "expected_products": ["Shaan Rejuvenation Cream"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q03",
        "question": "What ingredients are in Shaan Body Cream?",
        "category": "skin_care", "intent": "ingredients",
        "expected_products": ["Shaan Body Cream"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q04",
        "question": "What is Glamy Lab Matcha Plumping Gel used for?",
        "category": "skin_care", "intent": "usage",
        "expected_products": ["Glamy Lab Matcha Plumping Gel"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q05",
        "question": "What is the pregnancy safety classification of Starville Whitening Cream?",
        "category": "skin_care", "intent": "pregnancy_safety",
        "expected_products": ["Starville Whitening Cream"],
        "expected_labels": ["CONSULT A DOCTOR FIRST"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q06",
        "question": "Is CLARY Anti-Dandruff Shampoo safe during pregnancy?",
        "category": "hair_care", "intent": "pregnancy_safety",
        "expected_products": ["CLARY Anti-Dandruff Shampoo"],
        "expected_labels": ["CONSULT A DOCTOR FIRST"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q07",
        "question": "What are the main ingredients in Clary Hair Mask?",
        "category": "hair_care", "intent": "ingredients",
        "expected_products": ["Clary Hair Mask"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q08",
        "question": "What are the key benefits of Clary Hair Serum?",
        "category": "hair_care", "intent": "benefits",
        "expected_products": ["Clary Hair Serum"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q09",
        "question": "How should Clary Booster Shot be used?",
        "category": "hair_care", "intent": "usage",
        "expected_products": ["Clary Booster Shot"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q10",
        "question": "What is Seropipe Hair Dropper designed to help with?",
        "category": "hair_care", "intent": "benefits",
        "expected_products": ["Seropipe Hair Dropper"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q11",
        "question": "Which hair-care products are classified as AVOID DURING PREGNANCY?",
        "category": "hair_care", "intent": "pregnancy_safety",
        "expected_products": [],
        "expected_labels": ["AVOID DURING PREGNANCY"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q12",
        "question": "What does CONSULT A DOCTOR FIRST mean in the supplied pregnancy-safety reviews?",
        "category": None, "intent": "pregnancy_safety",
        "expected_products": [],
        "expected_labels": ["CONSULT A DOCTOR FIRST"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q13",
        "question": "Is Clary Hair Water safe during pregnancy?",
        "category": "hair_care", "intent": "pregnancy_safety",
        "expected_products": ["Clary Hair Water"],
        "expected_labels": ["CONSULT A DOCTOR FIRST"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q14",
        "question": "Is Shaan Nail Care safe during pregnancy?",
        "category": "skin_care", "intent": "pregnancy_safety",
        "expected_products": ["Shaan Nail Care"],
        "expected_labels": ["CONSULT A DOCTOR FIRST"],
        "must_be_in_scope": True,
    },
    {
        "id": "Q15",
        "question": "What ingredients are found in Glamy Lab Matcha Purifying Foam Cleanser?",
        "category": "skin_care", "intent": "ingredients",
        "expected_products": ["Glamy Lab Matcha Purifying Foam Cleanser"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q16",
        "question": "How often should Clary Scalp Scrub be used?",
        "category": "hair_care", "intent": "usage",
        "expected_products": ["Clary Scalp Scrub"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q17",
        "question": "Compare Shaan Cleanser and Starville Whitening Cleanser based on the supplied information.",
        "category": "skin_care", "intent": "comparison",
        "expected_products": ["Shaan Cleanser", "Starville Whitening Cleanser"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q18",
        "question": "Compare Clary Hair Mask and Seropipe Hair Mask.",
        "category": "hair_care", "intent": "comparison",
        "expected_products": ["Clary Hair Mask", "Seropipe Hair Mask"],
        "expected_labels": [],
        "must_be_in_scope": True,
    },
    {
        "id": "Q19",
        "question": "What blood tests should be ordered during pregnancy?",
        "category": None, "intent": "out_of_scope",
        "expected_products": [],
        "expected_labels": [],
        "must_be_in_scope": False,
    },
    {
        "id": "Q20",
        "question": "What medication should I take for severe hair loss during pregnancy?",
        "category": None, "intent": "out_of_scope",
        "expected_products": [],
        "expected_labels": [],
        "must_be_in_scope": False,
    },
]

assert len(GOLDEN_DATASET) == 20
print("Golden questions:", len(GOLDEN_DATASET))
display(pd.DataFrame(GOLDEN_DATASET))


Golden questions: 20


,id,question,category,intent,expected_products,expected_labels,must_be_in_scope
0,Q01,What is the pregnancy safety classification of...,skin_care,pregnancy_safety,[Shaan Cleanser],[SAFE],True
1,Q02,What are the key benefits of Shaan Rejuvenatio...,skin_care,benefits,[Shaan Rejuvenation Cream],[],True
2,Q03,What ingredients are in Shaan Body Cream?,skin_care,ingredients,[Shaan Body Cream],[],True
3,Q04,What is Glamy Lab Matcha Plumping Gel used for?,skin_care,usage,[Glamy Lab Matcha Plumping Gel],[],True
4,Q05,What is the pregnancy safety classification of...,skin_care,pregnancy_safety,[Starville Whitening Cream],[CONSULT A DOCTOR FIRST],True
5,Q06,Is CLARY Anti-Dandruff Shampoo safe during pre...,hair_care,pregnancy_safety,[CLARY Anti-Dandruff Shampoo],[CONSULT A DOCTOR FIRST],True
6,Q07,What are the main ingredients in Clary Hair Mask?,hair_care,ingredients,[Clary Hair Mask],[],True
7,Q08,What are the key benefits of Clary Hair Serum?,hair_care,benefits,[Clary Hair Serum],[],True
8,Q09,How should Clary Booster Shot be used?,hair_care,usage,[Clary Booster Shot],[],True
9,Q10,What is Seropipe Hair Dropper designed to help...,hair_care,benefits,[Seropipe Hair Dropper],[],True


## 19 — Evaluation metrics

In [27]:

def retrieval_product_hit(retrieval: Dict[str, Any], expected_products: List[str]) -> bool:
    if not expected_products:
        return True

    retrieved_products = {
        x[1]["metadata"].get("product", "").strip().lower()
        for x in retrieval["reranked"]
    }

    return all(
        any(ep.lower() == rp for rp in retrieved_products)
        for ep in expected_products
    )

def answer_contains_any_label(answer_text: str, labels: List[str]) -> bool:
    if not labels:
        return True
    low = answer_text.lower()
    return all(label.lower() in low for label in labels)

def evaluate_case(case: Dict[str, Any]) -> Dict[str, Any]:
    started = time.perf_counter()

    try:
        result = ask(case["question"], debug=True)
        latency = result.get("_trace", {}).get(
            "latency_ms",
            (time.perf_counter() - started) * 1000,
        )

        trace = result.get("_trace", {})
        retrieval = result.get("_retrieval")

        if case["must_be_in_scope"]:
            scope_correct = result.get("status") == "answered"
        else:
            scope_correct = result.get("status") == "out_of_scope"

        product_hit = (
            retrieval_product_hit(retrieval, case["expected_products"])
            if retrieval is not None else False
        )

        labels_ok = answer_contains_any_label(
            result.get("answer", ""),
            case["expected_labels"]
        )

        citation_ok = result.get("_guards", {}).get("citation", {}).get("valid", False)
        safety_ok = result.get("_guards", {}).get("safety", {}).get("valid", False)

        schema_ok = all(
            key in result
            for key in ["status", "answer", "evidence_summary", "citations", "confidence"]
        )

        return {
            "id": case["id"],
            "question": case["question"],
            "status": result.get("status"),
            "schema_valid": schema_ok,
            "scope_correct": scope_correct,
            "product_hit": product_hit,
            "expected_labels_present": labels_ok,
            "citations_valid": citation_ok if result.get("status") == "answered" else True,
            "safety_guard_valid": safety_ok if result.get("status") == "answered" else True,
            "confidence": result.get("confidence"),
            "latency_ms": round(float(latency), 2),
            "error": "",
        }

    except Exception as e:
        return {
            "id": case["id"],
            "question": case["question"],
            "status": "ERROR",
            "schema_valid": False,
            "scope_correct": False,
            "product_hit": False,
            "expected_labels_present": False,
            "citations_valid": False,
            "safety_guard_valid": False,
            "confidence": "insufficient",
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "error": repr(e),
        }

# Run after you have initialized the models + API key.
def run_evaluation(dataset=GOLDEN_DATASET):
    rows = []
    for case in tqdm(dataset):
        rows.append(evaluate_case(case))
    results_df = pd.DataFrame(rows)

    summary = {
        "cases": len(results_df),
        "schema_validity": float(results_df["schema_valid"].mean()),
        "scope_accuracy": float(results_df["scope_correct"].mean()),
        "product_retrieval_hit_rate": float(results_df["product_hit"].mean()),
        "expected_safety_label_rate": float(results_df["expected_labels_present"].mean()),
        "citation_validity": float(results_df["citations_valid"].mean()),
        "safety_guard_validity": float(results_df["safety_guard_valid"].mean()),
        "avg_latency_ms": float(results_df["latency_ms"].mean()),
        "p95_latency_ms": float(results_df["latency_ms"].quantile(0.95)),
    }

    return results_df, summary


In [28]:
def judge_faithfulness(question: str, answer: str, context: str) -> Dict[str, Any]:
    judge_prompt = f"""Question: {question}
Answer: {answer}
Context: {context}

Rate the answer from 1-5 on:
1. Faithfulness (is every claim supported by the context?)
2. Relevancy (does it actually address the question?)
Return ONLY JSON: {{"faithfulness": int, "relevancy": int, "reasoning": str}}"""

    raw = call_llm([{"role": "user", "content": judge_prompt}])
    return extract_json_object(raw)

## 20 — Optional retrieval-only benchmark

This benchmark does not call the LLM. It checks whether the retriever can put the expected product into the top-K context.

This is useful when tuning:
- embedding model
- dense/sparse K
- RRF
- reranking
- chunk size


In [29]:

def retrieval_benchmark(dataset=GOLDEN_DATASET, k=5):
    rows = []
    for case in tqdm(dataset):
        if case["intent"] == "out_of_scope":
            continue

        r = hybrid_retrieve(case["question"])
        topk = r["reranked"][:k]
        retrieved = [x[1]["metadata"].get("product", "") for x in topk]

        expected = case["expected_products"]
        if expected:
            hit = all(ep.lower() in {p.lower() for p in retrieved} for ep in expected)
        else:
            hit = True

        rows.append({
            "id": case["id"],
            "question": case["question"],
            "expected_products": expected,
            "retrieved_products": retrieved,
            f"hit@{k}": hit,
        })

    df = pd.DataFrame(rows)
    print(f"Hit@{k}: {df[f'hit@{k}'].mean():.3f}")
    return df


## 21 — Export artifacts / reproducibility

In [30]:

# Export the chunk manifest and a compact index manifest.
manifest_path = Path(CONFIG["artifact_dir"]) / "chunk_manifest.csv"
chunk_df.drop(columns=["text"]).to_csv(manifest_path, index=False)

index_manifest = {
    "assistant": CONFIG["assistant_name"],
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "sources": pdf_stats,
    "config": {k: v for k, v in CONFIG.items() if k not in {"artifact_dir", "chroma_dir"}},
    "chunk_count": len(chunks),
    "collection": CONFIG["chroma_collection"],
}

with open(Path(CONFIG["artifact_dir"]) / "index_manifest.json", "w", encoding="utf-8") as f:
    json.dump(index_manifest, f, ensure_ascii=False, indent=2, default=str)

print("Saved:")
print("-", manifest_path)
print("-", Path(CONFIG["artifact_dir"]) / "index_manifest.json")
print("-", Path(CONFIG["chroma_dir"]))


Saved:
- parkville_rag_artifacts/chunk_manifest.csv
- parkville_rag_artifacts/index_manifest.json
- parkville_rag_artifacts/chroma


In [31]:
!pip install -q gradio

import gradio as gr

def gradio_ask(question, history):
    result = ask(question)
    answer = result.get("answer", "")
    status = result.get("status", "")
    confidence = result.get("confidence", "")
    sources = ", ".join(c.get("product","") for c in result.get("citations", []) if c.get("product"))
    footer = f"\n\n---\n**Status:** {status} | **Confidence:** {confidence}"
    if sources:
        footer += f" | **Sources:** {sources}"
    return answer + footer

demo = gr.ChatInterface(
    fn=gradio_ask,
    title="Parkville Skin & Hair Care Expert",
    description="اسأل عن أي منتج Skin/Hair Care من كتالوج Parkville."
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://007c9c52fddc2c90a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 22 — Production checklist

Before calling the system production-ready, verify:

- [ ] Both PDFs extract cleanly
- [ ] Product detection is sensible
- [ ] Chunk manifest has correct product/section/page lineage
- [ ] Dense + BM25 retrieval both work
- [ ] Reranker is loaded
- [ ] Confidence gate refuses weak queries
- [ ] Citation guard rejects fabricated chunk IDs
- [ ] Pregnancy labels are preserved exactly from source evidence
- [ ] All 20 Golden questions are run
- [ ] Retrieval hit-rate is acceptable
- [ ] Scope/refusal accuracy is acceptable
- [ ] Latency and API error behavior are acceptable
- [ ] `index_manifest.json` is archived with the deployed index


## Engineering notes

1. **PDF-first ingestion:** runtime always reads the uploaded PDFs directly.
2. **No silent medical knowledge injection:** the generator is explicitly constrained to retrieved source evidence.
3. **Safety labels are treated as immutable source data.**
4. **Hybrid retrieval is deliberate:** dense search handles semantic similarity while BM25 protects exact product/ingredient recall.
5. **Reranking is narrow and expensive by design:** retrieve broadly, rerank the shortlist.
6. **The LLM is not the retrieval engine:** retrieval and guardrails decide what evidence is available; the LLM only verbalizes grounded evidence.
7. **The notebook is self-contained enough to evolve into a service:** the functions can later be moved into `ingestion/`, `retrieval/`, `guardrails/`, `api/`, and `evaluation/` modules without changing the architecture.
